# DPO gate + export -- from Google Drive (Colab)

For a DPO run whose training finished in `distill_dpo.ipynb` but whose Colab runtime is
gone. Everything it needs is on Drive (`final-adapter/`, `log_history.json`) or the HF
Hub (ckpt100, the pairs). It trains nothing.

**Why a new gate (2026-10-23).** dpo2 (lr 5e-5) failed the export gate in
`distill_dpo.ipynb`: at the first token where `chosen` and `rejected` differ, greedy
decoding picked `chosen`'s token in 4 of 39 val pairs, the same as ckpt100. That gate
counts the wrong event. `chosen` is `rejected` with the unsupported edge deleted, so at
the split `rejected` starts the removed edge and `chosen` starts the *next* edge. A model
that no longer writes the removed edge, but opens a different edge first, scored zero, and
a removed edge that came back later in the reply was never seen.

This notebook lets each model finish every val reply greedily from the split point and
counts edges by (source, target, relation):

- **removed edges written**: the unsupported edges the pair deleted (57 in the val set);
- **kept edges written**: the edges `chosen` keeps after the split (141);
- **valid JSON** replies.

**Pass bar, fixed before any result was seen:** the DPO model writes at most
`MAX_REMOVED_SHARE` (75%) as many removed edges as ckpt100, at least `MIN_KEPT_SHARE`
(90%) as many kept edges, and no fewer valid JSON replies. ckpt100 must itself write
at least `MIN_BASELINE_REMOVED` (10) removed edges, or the val set cannot show a reduction. Only then does it merge and
push to `HF_MODEL_REVISION`.

**Use:** HF token secret as before, check section 3, then **Runtime -> Run all** (L4).
Roughly 30-50 minutes, most of it the gate's generation.

## 1. Install packages

Same as `distill_dpo.ipynb` section 1.

In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys


def pip_install(*args: str) -> None:
    """Run pip, print ALL of its output, and raise on failure (never hidden)."""
    cmd = [sys.executable, "-m", "pip", "install", *args]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(proc.stdout, flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"pip install failed (exit {proc.returncode}): {' '.join(args)}")


if importlib.util.find_spec("unsloth") is None:
    pip_install("--upgrade", "pip")
    pip_install("unsloth")
    pip_install("trl", "datasets", "huggingface_hub")

for pkg in ("unsloth", "trl", "transformers", "peft", "datasets", "huggingface_hub"):
    print(f"{pkg}=={importlib.metadata.version(pkg)}")
print("packages ready")

## 1a. Fast causal_conv1d kernel -- ON here

`distill_dpo.ipynb` section 1a with `USE_HUB_CAUSAL_CONV1D = True`: this notebook only generates, and only the kernel's backward pass is broken (it wrecked dpo2's gradients).

In [ ]:
import importlib.util
import sys
from pathlib import Path

import torch
from huggingface_hub import snapshot_download

CC1D_REPO, CC1D_REVISION = "kernels-community/causal-conv1d", "v1"
_QWEN_MOD = "transformers.models.qwen3_5.modeling_qwen3_5"


def load_causal_conv1d():
    tv = "".join(torch.__version__.split("+")[0].split(".")[:2])
    variant = f"torch{tv}-cxx11-cu{torch.version.cuda.replace('.', '')}-x86_64-linux"
    root = Path(snapshot_download(CC1D_REPO, revision=CC1D_REVISION,
                                  allow_patterns=[f"build/{variant}/*"]))
    pkg = root / "build" / variant
    if not (pkg / "__init__.py").exists():
        raise FileNotFoundError(f"{CC1D_REPO}@{CC1D_REVISION} has no {variant} build")
    spec = importlib.util.spec_from_file_location(
        "causal_conv1d", pkg / "__init__.py", submodule_search_locations=[str(pkg)])
    mod = importlib.util.module_from_spec(spec)
    sys.modules["causal_conv1d"] = mod
    spec.loader.exec_module(mod)
    return mod, variant


def check_causal_conv1d(mod) -> None:
    # Compare against the reference implementations shipped in the same package.
    ref = mod.causal_conv1d_interface
    g = torch.Generator(device="cuda").manual_seed(0)
    x = torch.randn(2, 96, 333, device="cuda", dtype=torch.bfloat16, generator=g)
    w = torch.randn(96, 4, device="cuda", dtype=torch.bfloat16, generator=g)
    b = torch.randn(96, device="cuda", dtype=torch.bfloat16, generator=g)
    xk, xr = x.clone().requires_grad_(), x.float().clone().requires_grad_()
    out = mod.causal_conv1d_fn(xk, w, b, activation="silu")
    want = ref.causal_conv1d_ref(xr, w.float(), b.float(), activation="silu")
    assert torch.allclose(out.float(), want, atol=0.1, rtol=0.05), "forward mismatch"
    out.float().sum().backward()
    want.sum().backward()
    assert torch.allclose(xk.grad.float(), xr.grad, atol=0.1, rtol=0.05), "backward mismatch"
    state = torch.randn(2, 96, 4, device="cuda", dtype=torch.bfloat16, generator=g)
    step = torch.randn(2, 96, 1, device="cuda", dtype=torch.bfloat16, generator=g)
    s_k, s_r = state.clone(), state.float().clone()
    out = mod.causal_conv1d_update(step, s_k, w, b, activation="silu")
    want = ref.causal_conv1d_update_ref(step.float(), s_r, w.float(), b.float(), activation="silu")
    assert torch.allclose(out.float(), want, atol=0.1, rtol=0.05), "decode-step mismatch"
    assert torch.allclose(s_k.float(), s_r, atol=0.05), "decode-step state mismatch"


USE_HUB_CAUSAL_CONV1D = True    # dpo2: its BACKWARD gave NaN / ~5e5 gradient norms
CC1D_LOADED = False
if not USE_HUB_CAUSAL_CONV1D:
    print("causal_conv1d: hub kernel off (USE_HUB_CAUSAL_CONV1D = False) -- PyTorch path")
elif _QWEN_MOD in sys.modules:
    print("transformers' Qwen3.5 code is already imported, so a kernel registered now would "
          "be ignored -- Runtime -> Restart session, then Run all")
else:
    try:
        _cc1d, _variant = load_causal_conv1d()
        check_causal_conv1d(_cc1d)
        CC1D_LOADED = True
        print(f"causal_conv1d: {CC1D_REPO} {_variant} loaded and matches its reference")
    except Exception as e:
        for _k in [k for k in sys.modules if k.split(".")[0] == "causal_conv1d"]:
            del sys.modules[_k]
        print(f"causal_conv1d: hub kernel NOT used ({type(e).__name__}: {e}) -- "
              "training runs on the PyTorch path (correct, slower)")

## 2. HF token -- Colab secret or environment, never hardcoded

Set a Colab secret named `HF_TOKEN` (the key icon in the left sidebar) *or* export
`HF_TOKEN` in the environment before starting this runtime. This cell never prints the
token -- only whether one was found.

In [ ]:
import os


def get_hf_token() -> str:
    try:
        from google.colab import userdata  # type: ignore
        try:
            tok = userdata.get("HF_TOKEN")
            if tok:
                return tok
        except Exception:
            pass  # secret not set -- fall through to the environment
    except ImportError:
        pass  # not running on Colab
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not tok:
        raise RuntimeError(
            "no HF token found -- add a Colab secret named HF_TOKEN, or set the "
            "HF_TOKEN environment variable, before running this cell")
    return tok


HF_TOKEN = get_hf_token()
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF token loaded and logged in (value never printed)")

## 3. Configuration

The same configuration cell as `distill_dpo.ipynb`: `RUN_NAME` picks the Drive folder, and
`HF_MODEL_REVISION` the branch the export creates. The training settings in it are unused
here; the gate thresholds (`MAX_REMOVED_SHARE`, `MIN_KEPT_SHARE`,
`MIN_BASELINE_REMOVED`) are in it too.

In [ ]:
from pathlib import Path

# --- dataset: dpo_pairs.py output (train.jsonl / val.jsonl), "drive" or "hf_dataset" -------
DATA_SOURCE = "hf_dataset"
HF_DATASET_REPO = "carev01/ft-opus-distill"   # private
HF_DATASET_REVISION = "dpo1"
DRIVE_DATA_DIR = "/content/drive/MyDrive/ft-opus-dpo1"   # only when DATA_SOURCE == "drive"

# --- Google Drive: checkpoints, final adapter, metrics (survive a runtime death) ----------
RUN_NAME = "opus-distill-dpo3"     # dpo1: 5e-6, too weak; dpo2: 5e-5 on a broken kernel
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/graph-rag-distill") / RUN_NAME
CKPT_DIR = DRIVE_RUN_DIR / "checkpoints"
FINAL_ADAPTER_DIR = DRIVE_RUN_DIR / "final-adapter"
FINAL_MARKER = FINAL_ADAPTER_DIR / "COMPLETE"   # written LAST: adapter fully saved

# --- the model DPO starts from: the merged SFT checkpoint the pairs were generated with ---
HF_MODEL_REPO = "carev01/qwen35-4b-graphrag"
START_REVISION = "opus-distill-2026-10-ckpt100"
START_DIR = Path("/content/start-model")    # adapter_config points here: keep it fixed
MAX_SEQ_LENGTH = 20480
LOAD_IN_4BIT = True

# --- LoRA (same shape as the SFT run) ----------------------------------------------------
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

# --- DPO -----------------------------------------------------------------------------------
BETA = 0.1                 # strength of the pull towards the reference (ckpt100)
NLL_ALPHA = 1.0            # NLL on chosen (rpo_alpha / "sft" loss): protects coverage; 0 = plain DPO
LEARNING_RATE = 5e-5       # dpo1's 5e-6 moved the split-point log-odds by only +0.34
NUM_TRAIN_EPOCHS = 2
WARMUP_RATIO = 0.1
LR_SCHEDULER_TYPE = "cosine"
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
EVAL_STEPS = 10            # also the save interval, so the best checkpoint is on Drive
SAVE_TOTAL_LIMIT = 4
SEED = 20261021
GRAD_NORM_LIMIT = 1000.0   # training guard: stop on a larger norm from GRAD_NORM_FROM_STEP on
GRAD_NORM_FROM_STEP = 12   # dpo1: >100k for steps 1-10 (before the first eval), then <6

# --- export ----------------------------------------------------------------------------------
HF_MODEL_REVISION = "opus-distill-2026-10-dpo3"
MAX_REMOVED_SHARE = 0.75   # gate: DPO writes at most this share of ckpt100's removed edges
MIN_KEPT_SHARE = 0.90      # ... and at least this share of ckpt100's kept edges
MIN_BASELINE_REMOVED = 10  # ckpt100 must write this many removed edges, or nothing is measurable
HF_MODEL_PRIVATE = False   # the repo is public by design; a branch cannot be more private

OUT_DIR = Path("/content/outputs/opus-distill-dpo")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("config set")

## 3a. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
_ckpts = sorted(CKPT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
print(f"run dir: {DRIVE_RUN_DIR}")
print(f"checkpoints on Drive: {[p.name for p in _ckpts] or 'none'}")
print(f"final adapter on Drive (complete): {FINAL_MARKER.exists()}")

## 4. Get the pairs

In [ ]:
DATA_DIR = Path("/content/data/ft-opus-dpo1")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if DATA_SOURCE == "drive":
    DATA_DIR = Path(DRIVE_DATA_DIR)
elif DATA_SOURCE == "hf_dataset":
    from huggingface_hub import hf_hub_download
    for fname in ("train.jsonl", "val.jsonl", "MANIFEST.json", "sample.json"):
        try:
            p = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                                filename=fname, revision=HF_DATASET_REVISION, token=HF_TOKEN)
        except Exception:
            if fname in ("sample.json", "MANIFEST.json"):
                print(f"  {fname} not in {HF_DATASET_REPO}@{HF_DATASET_REVISION} -- skipping")
                continue
            raise
        (DATA_DIR / fname).write_bytes(Path(p).read_bytes())
else:
    raise ValueError(f"unknown DATA_SOURCE {DATA_SOURCE!r}")

assert (DATA_DIR / "train.jsonl").exists() and (DATA_DIR / "val.jsonl").exists(), DATA_DIR
print(f"dataset ready at {DATA_DIR}")

In [ ]:
import json
from collections import Counter


def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


train_records = read_jsonl(DATA_DIR / "train.jsonl")
val_records = read_jsonl(DATA_DIR / "val.jsonl")
for _r in train_records + val_records:
    assert [m["role"] for m in _r["messages"]] == ["system", "user"], _r["meta"]
    _c, _x = json.loads(_r["chosen"])["edges"], json.loads(_r["rejected"])["edges"]
    assert 0 < len(_c) < len(_x), _r["meta"]
    assert all(e in _x for e in _c), _r["meta"]
_cats = Counter(c for r in train_records for f in r["meta"]["removed"] for c in f["categories"][:1])
print(f"train pairs {len(train_records)}, val pairs {len(val_records)}")
print(f"facts removed (train): {sum(len(r['meta']['removed']) for r in train_records)}; "
      f"categories {dict(_cats)}")
assert len(train_records) >= 50, "too few pairs for a DPO round -- capture more first"

## 5. Download ckpt100, load it and the DPO model

The adapter on Drive records `START_DIR` as its base, so ckpt100 goes there first. Both
models load in 4-bit, as in training; the comparison is like for like.

In [ ]:
from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel
import torch

assert FINAL_MARKER.exists(), (f"no complete final adapter at {FINAL_ADAPTER_DIR} -- "
                               "is RUN_NAME right and Drive mounted?")
snapshot_download(repo_id=HF_MODEL_REPO, revision=START_REVISION, local_dir=str(START_DIR),
                  ignore_patterns=["*.gguf"], token=HF_TOKEN)
assert any(START_DIR.glob("*.safetensors")), f"no safetensors in {HF_MODEL_REPO}@{START_REVISION}"
_base_ref = json.loads((FINAL_ADAPTER_DIR / "adapter_config.json").read_text())["base_model_name_or_path"]
assert Path(_base_ref) == START_DIR, f"adapter's base is {_base_ref}, not {START_DIR}"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(FINAL_ADAPTER_DIR), max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT, dtype=None)
FastLanguageModel.for_inference(model)
base, base_tok = FastLanguageModel.from_pretrained(
    model_name=str(START_DIR), max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT, dtype=None)
FastLanguageModel.for_inference(base)
print(f"loaded: DPO = {START_REVISION} + {FINAL_ADAPTER_DIR}; ckpt100 = {START_DIR}")
import transformers.models.qwen3_5.modeling_qwen3_5 as _mq


def calls_package(fn, package: str) -> bool:
    # Follow transformers' kernel wrappers down to the implementation they call.
    seen, stack = set(), [fn]
    while stack:
        f = stack.pop()
        if f is None or id(f) in seen or not callable(f):
            continue
        seen.add(id(f))
        if str(getattr(f, "__module__", "")).split(".")[0] == package:
            return True
        stack.append(getattr(f, "__wrapped__", None))
        for c in getattr(f, "__closure__", None) or ():
            try:
                stack.append(c.cell_contents)
            except ValueError:
                pass
    return False


_fast = {"causal_conv1d": calls_package(_mq.causal_conv1d_fn, "causal_conv1d"),
         "fla": calls_package(getattr(_mq, "torch_chunk_gated_delta_rule", None), "fla")}
print(f"fast kernels in use: {_fast}")
assert USE_HUB_CAUSAL_CONV1D or not _fast["causal_conv1d"], (
    "the hub causal_conv1d kernel is off, yet the model calls a causal_conv1d package -- "
    "its backward pass broke dpo2; restart the session without it")
if CC1D_LOADED and not _fast["causal_conv1d"]:
    print("WARNING: the causal_conv1d kernel loaded in section 1a but the model does not "
          "call it -- this run is on the PyTorch path")

## 6. Prompt formatting

Same as `distill_dpo.ipynb` section 7: the serving prompt.

In [ ]:
from datasets import Dataset

END_MARK = "<|im_end|>"
# Unsloth may hand back a multimodal processor for this architecture; its __call__ batches
# (len(ids) would be 1) and it has no eos_token. Count and decode with the text tokenizer.
TEXT_TOK = getattr(tokenizer, "tokenizer", tokenizer)
RESPONSE_PART = tokenizer.apply_chat_template(
    [{"role": "user", "content": "x"}], tokenize=False, add_generation_prompt=True,
    enable_thinking=False).rsplit(END_MARK + "\n", 1)[-1]
print(f"serving prompt ends with {RESPONSE_PART!r}")


def to_pair(r: dict, mark: str) -> dict:
    prompt = tokenizer.apply_chat_template(r["messages"], tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
    assert prompt.endswith(RESPONSE_PART), prompt[-80:]
    return {"prompt": prompt, "chosen": r["chosen"] + mark, "rejected": r["rejected"] + mark}


def build(records: list[dict], mark: str) -> Dataset:
    rows, dropped = [], 0
    for r in records:
        p = to_pair(r, mark)
        n = len(TEXT_TOK(p["prompt"] + p["rejected"], add_special_tokens=False)["input_ids"])
        if n > MAX_SEQ_LENGTH:
            dropped += 1
            continue
        rows.append(p)
    print(f"  {len(rows)} pairs kept, {dropped} over {MAX_SEQ_LENGTH} tokens dropped")
    return Dataset.from_list(rows)


_lens = sorted(len(TEXT_TOK(to_pair(r, "")["prompt"], add_special_tokens=False)["input_ids"])
               for r in train_records)
MAX_PROMPT_TOKENS = min(_lens[-1], MAX_SEQ_LENGTH)
print(f"prompt tokens: median {_lens[len(_lens) // 2]}, max {_lens[-1]}")

## 7. What the optimiser did

From `log_history.json` on Drive: the learning rate should peak at `LEARNING_RATE`, and the
gradient norm should be finite and non-zero at every step (dpo2's eval rows at steps 20 and
30 were identical to every printed digit).

In [ ]:
_hist = json.loads((DRIVE_RUN_DIR / "log_history.json").read_text())
_train = [h for h in _hist if "loss" in h and "grad_norm" in h]
for h in _train:
    print(f"step {h['step']:>3}  lr {h.get('learning_rate', 0):.2e}  "
          f"grad_norm {h['grad_norm']:.4f}  loss {h['loss']:.4f}")
for h in (h for h in _hist if "eval_loss" in h):
    print(f"eval @ {h['step']:>3}: loss {h['eval_loss']:.6f}  "
          f"margins {h.get('eval_rewards/margins', float('nan')):.4f}")
_peak = max(h.get("learning_rate", 0) for h in _train)
_dead = [h["step"] for h in _train if not h["grad_norm"] or h["grad_norm"] != h["grad_norm"]]
print(f"peak lr {_peak:.2e} (configured {LEARNING_RATE:.0e}); steps with zero/NaN grad norm: {_dead}")
if abs(_peak - LEARNING_RATE) > 0.1 * LEARNING_RATE:
    print("WARNING: the learning rate never reached the configured value")

## 8. Gate: does the DPO model stop writing the removed edges?

For each val pair, both models continue greedily from the shared prefix up to the split
and finish the reply. `starts_removed` is the old token-level view (does the very next
token begin the removed edge); the gate itself uses the edge counts.

In [ ]:
import re
import time

_TRIPLE = re.compile(r'"source_entity_name": "(.*?)", "target_entity_name": "(.*?)", "relation_type": "(.*?)"')


def _triples(text: str) -> tuple[set, bool]:
    try:
        return {(e["source_entity_name"], e["target_entity_name"], e["relation_type"])
                for e in json.loads(text)["edges"]}, True
    except (ValueError, KeyError, TypeError):
        return set(_TRIPLE.findall(text)), False


@torch.inference_mode()
def rest_from_split(m, r: dict) -> dict | None:
    p = to_pair(r, "")
    n_prompt = len(TEXT_TOK(p["prompt"], add_special_tokens=False)["input_ids"])
    ch = TEXT_TOK(p["prompt"] + p["chosen"], add_special_tokens=False)["input_ids"]
    rj = TEXT_TOK(p["prompt"] + p["rejected"], add_special_tokens=False)["input_ids"]
    split = next((i for i, (a, b) in enumerate(zip(ch, rj)) if a != b), None)
    if split is None:
        return None
    ids = torch.tensor([ch[:split]], device=m.device)
    out = m.generate(input_ids=ids, attention_mask=torch.ones_like(ids), do_sample=False,
                     max_new_tokens=min(len(rj) - split + 256, 4096))
    new = out[0][split:].tolist()
    prefix = TEXT_TOK.decode(ch[n_prompt:split], skip_special_tokens=True)
    gen, ok = _triples(prefix + TEXT_TOK.decode(new, skip_special_tokens=True))
    done = set(_TRIPLE.findall(prefix))            # edges already complete before the split
    removed = _triples(r["rejected"])[0] - _triples(r["chosen"])[0]
    kept = _triples(r["chosen"])[0] - done
    return {"removed": len(removed), "removed_written": len(removed & gen),
            "kept": len(kept), "kept_written": len(kept & gen), "json": ok,
            "starts_removed": bool(new) and new[0] == rj[split]}


_tot, _per_pair = {}, {}
for _label, _m in (("ckpt100", base), ("dpo", model)):
    _t0 = time.time()
    _per_pair[_label] = []
    for _i, _r in enumerate(val_records, 1):
        _per_pair[_label].append(rest_from_split(_m, _r))
        if _i % 5 == 0 or _i == len(val_records):
            print(f"  [{_label}] {_i}/{len(val_records)} pairs, {time.time() - _t0:.0f}s", flush=True)
    _rows = [x for x in _per_pair[_label] if x]
    _tot[_label] = {k: sum(x[k] for x in _rows) for k in _rows[0]}
    print(f"[{_label}] {len(_rows)} pairs, {time.time() - _t0:.0f}s: {_tot[_label]}")
(DRIVE_RUN_DIR / "gate_continuation.json").write_text(
    json.dumps({"totals": _tot, "per_pair": _per_pair}, indent=1))

_b, _d = _tot["ckpt100"], _tot["dpo"]
print(f"removed edges written: dpo {_d['removed_written']} vs ckpt100 {_b['removed_written']} "
      f"(of {_b['removed']}); kept: dpo {_d['kept_written']} vs {_b['kept_written']} "
      f"(of {_b['kept']}); JSON: dpo {_d['json']} vs {_b['json']}")
assert _b["removed_written"] >= MIN_BASELINE_REMOVED, (
    f"ckpt100 writes only {_b['removed_written']} of the removed edges greedily -- too few "
    "for this val set to measure a reduction; the gate cannot decide")
GATE_PASSED = (_d["removed_written"] <= MAX_REMOVED_SHARE * _b["removed_written"]
               and _d["kept_written"] >= MIN_KEPT_SHARE * _b["kept_written"]
               and _d["json"] >= _b["json"])
assert GATE_PASSED, "gate failed -- not exporting (results saved to gate_continuation.json on Drive)"
print("GATE PASSED")

## 9. Smoke test, merge, push

Same as `distill_dpo.ipynb` section 10 after its gate: refuses a branch that already has
its own commits, checks three short held-out prompts still come back as JSON, merges to
bf16, uploads, and checks the Hub's shards equal the merged index.

In [ ]:
import gc
import json as _json

from huggingface_hub import HfApi

assert globals().get("GATE_PASSED"), "run the gate first; export only after it passes"
api = HfApi(token=HF_TOKEN)
# A new branch is a copy of main, weights included, so "the branch holds weights" cannot
# be the test: refuse only a branch that has moved away from main (an earlier export).
_refs = {b.name: b.target_commit
         for b in api.list_repo_refs(HF_MODEL_REPO, repo_type="model").branches}
assert _refs.get(HF_MODEL_REVISION, _refs["main"]) == _refs["main"], (
    f"{HF_MODEL_REPO}@{HF_MODEL_REVISION} already has its own commits (an earlier export?)"
    " -- choose a new HF_MODEL_REVISION")
api.create_branch(repo_id=HF_MODEL_REPO, repo_type="model", branch=HF_MODEL_REVISION,
                  exist_ok=True)


def edges_of(m, t, prompt: str) -> int | None:
    ids = t(text=prompt, return_tensors="pt").to(m.device)
    out = m.generate(**ids, max_new_tokens=4096, do_sample=False)   # 1024 cut an 11-edge reply
    reply = t.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        return len(_json.loads(reply)["edges"])
    except (ValueError, KeyError, TypeError):
        print(f"  not JSON: {reply[:160]!r} ... {reply[-160:]!r}")
        return None


for _r in sorted(val_records, key=lambda r: len(r["messages"][1]["content"]))[:3]:
    _prompt = to_pair(_r, "")["prompt"]
    _n_dpo, _n_base = edges_of(model, tokenizer, _prompt), edges_of(base, base_tok, _prompt)
    print(f"edges: dpo {_n_dpo}, ckpt100 {_n_base}, chosen {len(_json.loads(_r['chosen'])['edges'])}")
    assert _n_dpo is not None, "the DPO model did not answer in JSON -- do not upload this export"
print("smoke test passed")
del base, base_tok
gc.collect()
torch.cuda.empty_cache()

merged_dir = OUT_DIR / "merged-safetensors"
model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
_index = json.loads((merged_dir / "model.safetensors.index.json").read_text())
_missing = sorted(set(_index["weight_map"].values())
                  - {p.name for p in merged_dir.glob("*.safetensors")})
assert not _missing, f"merged save is missing shards: {_missing}"
api.upload_folder(repo_id=HF_MODEL_REPO, repo_type="model", folder_path=str(merged_dir),
                  revision=HF_MODEL_REVISION,
                  delete_patterns=["*.safetensors", "*.gguf"],   # no shard of main's survives
                  commit_message=f"{RUN_NAME}: DPO on {START_REVISION}, merged bf16")
_pushed = {f for f in api.list_repo_files(HF_MODEL_REPO, repo_type="model",
                                          revision=HF_MODEL_REVISION)
           if f.endswith((".safetensors", ".gguf"))}
assert _pushed == set(_index["weight_map"].values()), f"Hub shards != merged index: {_pushed}"
print(f"DONE: {HF_MODEL_REPO}@{HF_MODEL_REVISION}")

## 10. Next -- outside this notebook

Hand the revision name back. It is validated like ckpt100, on the SAME 60 trial articles:
held-out eval (`scripts/eval_finetune.py --guided`), the 45-chunk faithfulness check,
then `scripts/haiku_extract_ab.py --arms v2 --reuse <adoption-trial rows>`, judged twice.